# Data Exploration und Cleaning

Dieses Notebook liest die SMS-Spam-Rohdaten ein, prüft Datenqualität, Klassenverteilung, Encoding-Themen, HTML-Entities und Duplikate. Am Ende wird ein deduplizierter CSV-Datensatz nach `data/01_cleaned/sms_cleaned.csv` geschrieben.

In [2]:
from pathlib import Path
import html
import re

import pandas as pd
from IPython.display import display

RAW_PATH = Path("data/00_raw/SMSSpamCollection")
CLEAN_DIR = Path("data/01_cleaned")
CLEAN_PATH = CLEAN_DIR / "sms_cleaned.csv"

pd.set_option("display.max_colwidth", 140)

## 1. Rohdaten einlesen

Die Datei ist tab-getrennt und enthält keinen Header. Sie wird explizit als UTF-8 gelesen, damit Zeichen wie `£`, `ü` oder typografische Apostrophe korrekt erhalten bleiben.

In [3]:
df_raw = pd.read_csv(
    RAW_PATH,
    sep="\t",
    header=None,
    names=["label", "message"],
    encoding="utf-8",
)

display(df_raw.head(10))
print(f"Original rows: {len(df_raw)}")

,label,message
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply ...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives around here though"
5,spam,"FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, £..."
6,ham,Even my brother is not like to speak with me. They treat me like aids patent.
7,ham,As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press *9 to copy...
8,spam,WINNER!! As a valued network customer you have been selected to receivea £900 prize reward! To claim call 09061701461. Claim code KL341....
9,spam,Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co F...


Original rows: 5572


## 2. Labels normalisieren, HTML-Entities dekodieren und Target erzeugen

`html.unescape` dekodiert HTML-Entities wie `&lt;`, `&gt;` oder `&#gt;`. Das ist kein UTF-8-Mojibake, sondern HTML-Encoding innerhalb einzelner SMS-Texte.

In [4]:
df = df_raw.copy()
df["label"] = df["label"].astype("string").str.strip().str.lower()
df["message"] = df["message"].astype("string").str.strip()
df["message"] = df["message"].map(lambda value: html.unescape(value) if pd.notna(value) else value)
df["target"] = df["label"].map({"ham": 0, "spam": 1}).astype("Int64")
df = df[["label", "target", "message"]]

display(df.head(10))
display(df.dtypes.to_frame("dtype"))

,label,target,message
0,ham,0,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,ham,0,Ok lar... Joking wif u oni...
2,spam,1,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply ...
3,ham,0,U dun say so early hor... U c already then say...
4,ham,0,"Nah I don't think he goes to usf, he lives around here though"
5,spam,1,"FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, £..."
6,ham,0,Even my brother is not like to speak with me. They treat me like aids patent.
7,ham,0,As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press *9 to copy...
8,spam,1,WINNER!! As a valued network customer you have been selected to receivea £900 prize reward! To claim call 09061701461. Claim code KL341....
9,spam,1,Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co F...


,dtype
label,string
target,Int64
message,str


## 3. Anzahl und Klassenverteilung

In [5]:
class_distribution = (
    df["label"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .reset_index(name="count")
)
class_distribution["percentage"] = (class_distribution["count"] / len(df) * 100).round(2)
display(class_distribution)

print("Expected from dataset readme: 5,574 rows, 4,827 ham, 747 spam.")
print(f"Actual: {len(df)} rows")

,label,count,percentage
0,ham,4825,86.59
1,spam,747,13.41


Expected from dataset readme: 5,574 rows, 4,827 ham, 747 spam.
Actual: 5572 rows


Die Klassen sind deutlich unausgeglichen. SPAM macht nur etwa 13,4% der Originaldaten aus. Für spätere Modelle sind deshalb Precision, Recall und F1 für SPAM wichtiger als reine Accuracy.

## 4. Missing Values, leere Texte und ungültige Targets

In [6]:
missing_summary = pd.DataFrame(
    {
        "missing_values": df[["label", "target", "message"]].isna().sum(),
        "empty_or_whitespace": [
            df["label"].astype("string").str.strip().eq("").sum(),
            0,
            df["message"].astype("string").str.strip().eq("").sum(),
        ],
    },
    index=["label", "target", "message"],
)
display(missing_summary)

problem_rows = df[
    df["label"].isna()
    | df["target"].isna()
    | df["message"].isna()
    | df["label"].astype("string").str.strip().eq("")
    | df["message"].astype("string").str.strip().eq("")
]

print(f"Rows with missing/empty label, target or message: {len(problem_rows)}")
display(problem_rows.head(20))

,missing_values,empty_or_whitespace
label,0,0
target,0,0
message,0,0


Rows with missing/empty label, target or message: 0


,label,target,message


## 5. Encoding-, ASCII- und HTML-Entity-Checks

Nicht-ASCII-Zeichen sind in SMS normal, z. B. `£` oder `ü`. Problematisch sind Mojibake-Marker wie `Â`, `Ã` oder `â`, wenn sie durch falsches Decoding entstanden sind. Zusätzlich prüfen wir, ob nach `html.unescape` noch HTML-Entities wie `&lt;` oder `&#gt;` übrig sind.

In [7]:
mojibake_markers = ["Ã", "Â", "â€", "â€™", "â€œ", "â€�", "â€“", "â€”"]
mojibake_pattern = re.compile("|".join(re.escape(marker) for marker in mojibake_markers))
literal_escape_pattern = re.compile(r"\\x[0-9A-Fa-f]{2}|\\u[0-9A-Fa-f]{4}")
html_entity_pattern = re.compile(r"&(?:[a-zA-Z][a-zA-Z0-9]+|#[0-9]+|#x[0-9A-Fa-f]+);")
non_ascii_pattern = re.compile(r"[^\x00-\x7F]")

encoding_checks = df.assign(
    has_non_ascii=df["message"].str.contains(non_ascii_pattern, na=False),
    has_mojibake=df["message"].str.contains(mojibake_pattern, na=False),
    has_literal_escape=df["message"].str.contains(literal_escape_pattern, na=False),
    has_html_entity=df["message"].str.contains(html_entity_pattern, na=False),
)

encoding_summary = pd.DataFrame(
    {
        "check": [
            "messages with non-ASCII characters",
            "messages with likely mojibake markers",
            "messages with literal ASCII escape sequences",
            "messages with remaining HTML entities",
        ],
        "count": [
            int(encoding_checks["has_non_ascii"].sum()),
            int(encoding_checks["has_mojibake"].sum()),
            int(encoding_checks["has_literal_escape"].sum()),
            int(encoding_checks["has_html_entity"].sum()),
        ],
    }
)
display(encoding_summary)

display(encoding_checks.loc[encoding_checks["has_non_ascii"], ["label", "message"]].head(15))

mojibake_examples = encoding_checks.loc[encoding_checks["has_mojibake"], ["label", "message"]].head(20)
html_entity_examples = encoding_checks.loc[encoding_checks["has_html_entity"], ["label", "message"]].head(20)

print(f"Likely mojibake examples: {len(mojibake_examples)} shown")
display(mojibake_examples)
print(f"Remaining HTML entity examples: {len(html_entity_examples)} shown")
display(html_entity_examples)

,check,count
0,messages with non-ASCII characters,483
1,messages with likely mojibake markers,0
2,messages with literal ASCII escape sequences,0
3,messages with remaining HTML entities,0


,label,message
5,spam,"FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, £..."
8,spam,WINNER!! As a valued network customer you have been selected to receivea £900 prize reward! To claim call 09061701461. Claim code KL341....
12,spam,"URGENT! You have won a 1 week FREE membership in our £100,000 Prize Jackpot! Txt the word: CLAIM to No: 81010 T&C www.dbuk.net LCCLTD PO..."
18,ham,Fine if thats the way u feel. Thats the way its gota b
19,spam,"England v Macedonia - dont miss the goals/team news. Txt ur national team to 87077 eg ENGLAND to 87077 Try:WALES, SCOTLAND 4txt/ú1.20 PO..."
21,ham,I‘m going to try for 2 months ha ha only joking
22,ham,So ü pay first lar... Then when is da stock comin...
34,spam,Thanks for your subscription to Ringtone UK your mobile will be charged £5/month Please confirm by replying YES or NO. If you reply NO y...
35,ham,Yup... Ok i go home look at the timings then i msg ü again... Xuhui going to learn on 2nd may too but her lesson is at 8am
65,spam,"As a valued customer, I am pleased to advise you that following recent review of your Mob No. you are awarded with a £1500 Bonus Prize, ..."


Likely mojibake examples: 0 shown


,label,message


Remaining HTML entity examples: 0 shown


,label,message


## 6. Duplikate prüfen und entfernen

Duplikate werden nach dem bereinigten Nachrichtentext erkannt. Wenn ein identischer Text mehrfach vorkommt, behalten wir die erste Zeile und entfernen die weiteren Vorkommen.

In [8]:
duplicate_rows = df[df.duplicated(subset="message", keep=False)].copy()

duplicate_summary = (
    duplicate_rows.groupby("message")
    .agg(
        occurrences=("message", "size"),
        labels=("label", lambda labels: ", ".join(sorted(labels.unique()))),
    )
    .reset_index()
    .sort_values(["occurrences", "message"], ascending=[False, True])
)

label_conflicts = duplicate_summary[duplicate_summary["labels"].str.contains(",", na=False)]
duplicate_extra_rows = int(duplicate_summary["occurrences"].sub(1).sum()) if len(duplicate_summary) else 0

print(f"Duplicate groups: {len(duplicate_summary)}")
print(f"Duplicate rows to remove: {duplicate_extra_rows}")
print(f"Duplicate texts with label conflicts: {len(label_conflicts)}")

display(duplicate_summary.head(20))
display(label_conflicts.head(20))

df_clean = df.drop_duplicates(subset="message", keep="first").reset_index(drop=True)
print(f"Rows before deduplication: {len(df)}")
print(f"Rows after deduplication: {len(df_clean)}")

Duplicate groups: 289
Duplicate rows to remove: 414
Duplicate texts with label conflicts: 0


,message,occurrences,labels
207,"Sorry, I'll call later",30,ham
105,I cant pick the phone right now. Pls send a message,12,ham
174,Ok...,10,ham
4,"7 wonders in My WORLD 7th You 6th Ur style 5th Ur smile 4th Ur Personality 3rd Ur Nature 2nd Ur SMS and 1st ""Ur Lovely Friendship""... go...",4,ham
166,Ok,4,ham
173,Ok.,4,ham
176,Okie,4,ham
181,Please call our customer service representative on FREEPHONE 0808 145 4742 between 9am-11pm as you have WON a guaranteed £1000 cash or £...,4,spam
195,"Say this slowly.? GOD,I LOVE YOU & I NEED YOU,CLEAN MY HEART WITH YOUR BLOOD.Send this to Ten special people & u c miracle tomorrow, do ...",4,ham
256,"Wen ur lovable bcums angry wid u, dnt take it seriously.. Coz being angry is d most childish n true way of showing deep affection, care ...",4,ham


,message,occurrences,labels


Rows before deduplication: 5572
Rows after deduplication: 5158


## 7. Clean CSV schreiben

In [9]:
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(CLEAN_PATH, index=False, encoding="utf-8")
print(f"Saved: {CLEAN_PATH}")

df_check = pd.read_csv(CLEAN_PATH, encoding="utf-8")
display(df_check.head())
print(f"Rows written: {len(df_check)}")
print(f"Missing values after reload: {int(df_check[['label', 'target', 'message']].isna().sum().sum())}")

Saved: data\01_cleaned\sms_cleaned.csv


,label,target,message
0,ham,0,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,ham,0,Ok lar... Joking wif u oni...
2,spam,1,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply ...
3,ham,0,U dun say so early hor... U c already then say...
4,ham,0,"Nah I don't think he goes to usf, he lives around here though"


Rows written: 5158
Missing values after reload: 0


## 8. Original vs. Cleaned nebeneinander

Diese Übersicht zeigt, wie sich das Cleaning auf Zeilenanzahl, Klassenverteilung, Missing Values, HTML-Entities und Duplikate auswirkt.

In [10]:
def dataset_profile(dataframe, name):
    return pd.Series(
        {
            "rows": len(dataframe),
            "ham": int((dataframe["label"] == "ham").sum()),
            "spam": int((dataframe["label"] == "spam").sum()),
            "missing_label": int(dataframe["label"].isna().sum()),
            "missing_target": int(dataframe["target"].isna().sum()),
            "missing_message": int(dataframe["message"].isna().sum()),
            "remaining_html_entities": int(dataframe["message"].astype("string").str.contains(html_entity_pattern, na=False).sum()),
            "likely_mojibake_markers": int(dataframe["message"].astype("string").str.contains(mojibake_pattern, na=False).sum()),
            "duplicate_extra_rows": int(dataframe.duplicated(subset="message").sum()),
        },
        name=name,
    )

comparison = pd.concat(
    [dataset_profile(df, "original_after_basic_cleaning"), dataset_profile(df_clean, "cleaned_written_to_csv")],
    axis=1,
)
display(comparison)

class_comparison = pd.concat(
    [
        df["label"].value_counts().rename("original_after_basic_cleaning"),
        df_clean["label"].value_counts().rename("cleaned_written_to_csv"),
    ],
    axis=1,
)
class_comparison["removed"] = class_comparison["original_after_basic_cleaning"] - class_comparison["cleaned_written_to_csv"]
display(class_comparison)

,original_after_basic_cleaning,cleaned_written_to_csv
rows,5572,5158
ham,4825,4516
spam,747,642
missing_label,0,0
missing_target,0,0
missing_message,0,0
remaining_html_entities,0,0
likely_mojibake_markers,0,0
duplicate_extra_rows,414,0


,original_after_basic_cleaning,cleaned_written_to_csv,removed
label,,,
ham,4825,4516,309
spam,747,642,105
